In [1]:
import os
import time
import cv2
import numpy as np
import face_recognition as fr

# CONFIG
KNOWN_PEOPLE_DIR = "known_people"

SCALE_FACTOR = 0.5
TOLERANCE = 0.55

BOX_COLOR = (255, 0, 0)
TEXT_COLOR = (255, 255, 255)

# STORAGE
known_face_encodings = []
known_face_names = []

# LOAD FACES
def load_known_faces():

    for person_name in os.listdir(KNOWN_PEOPLE_DIR):

        person_path = os.path.join(
            KNOWN_PEOPLE_DIR,
            person_name
        )

        if not os.path.isdir(person_path):
            continue

        person_encodings = []

        print(f"\n[INFO] Loading {person_name}")

        for file_name in os.listdir(person_path):

            image_path = os.path.join(
                person_path,
                file_name
            )

            try:

                image = fr.load_image_file(image_path)

                encodings = fr.face_encodings(image)

                if len(encodings) == 0:
                    print(f"[WARNING] No face in {file_name}")
                    continue

                person_encodings.append(encodings[0])

                print(f"[SUCCESS] {file_name}")

            except Exception as e:
                print(f"[ERROR] {file_name}: {e}")

        # AVERAGE ENCODINGS
        if len(person_encodings) > 0:

            mean_encoding = np.mean(
                person_encodings,
                axis=0
            )

            known_face_encodings.append(mean_encoding)

            known_face_names.append(person_name)

            print(
                f"[INFO] {person_name} loaded with "
                f"{len(person_encodings)} images"
            )

# MAIN
def main():

    load_known_faces()

    if len(known_face_encodings) == 0:
        print("[ERROR] No faces loaded")
        return

    video_capture = cv2.VideoCapture(0)

    if not video_capture.isOpened():
        print("[ERROR] Webcam not available")
        return

    process_this_frame = True

    face_locations = []
    face_names = []

    # FPS
    prev_time = 0

    while True:

        ret, frame = video_capture.read()

        if not ret:
            print("[ERROR] Frame capture failed")
            break

        # FPS CALCULATION
        current_time = time.time()

        fps = 0

        if prev_time != 0:
            fps = 1 / (current_time - prev_time)

        prev_time = current_time

        # PROCESS FRAME
        if process_this_frame:

            small_frame = cv2.resize(
                frame,
                (0, 0),
                fx=SCALE_FACTOR,
                fy=SCALE_FACTOR
            )

            rgb_small_frame = cv2.cvtColor(
                small_frame,
                cv2.COLOR_BGR2RGB
            )

            face_locations = fr.face_locations(
                rgb_small_frame,
                model="hog"
            )

            face_encodings = fr.face_encodings(
                rgb_small_frame,
                face_locations
            )

            face_names = []

            print(
                f"\n[DEBUG] Faces detected: "
                f"{len(face_locations)}"
            )

            for face_encoding in face_encodings:

                face_distances = fr.face_distance(
                    known_face_encodings,
                    face_encoding
                )

                best_match_index = np.argmin(
                    face_distances
                )

                best_distance = face_distances[
                    best_match_index
                ]

                name = "Unknown"

                confidence = round(
                    (1 - best_distance) * 100,
                    2
                )

                if best_distance < TOLERANCE:

                    name = (
                        f"{known_face_names[best_match_index]}"
                        f" ({confidence}%)"
                    )

                print(
                    f"[DEBUG] Match: {name} | "
                    f"Distance: {best_distance}"
                )

                face_names.append(name)

        process_this_frame = not process_this_frame

        # DRAW RESULTS
        for (top, right, bottom, left), name in zip(
            face_locations,
            face_names
        ):

            top = int(top / SCALE_FACTOR)
            right = int(right / SCALE_FACTOR)
            bottom = int(bottom / SCALE_FACTOR)
            left = int(left / SCALE_FACTOR)

            # FACE BOX
            cv2.rectangle(
                frame,
                (left, top),
                (right, bottom),
                BOX_COLOR,
                2
            )

            # NAME BOX
            cv2.rectangle(
                frame,
                (left, bottom - 35),
                (right, bottom),
                BOX_COLOR,
                cv2.FILLED
            )

            # NAME TEXT
            cv2.putText(
                frame,
                name,
                (left + 6, bottom - 6),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                TEXT_COLOR,
                1
            )

        # SHOW FPS
        cv2.putText(
            frame,
            f"FPS: {int(fps)}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        # SHOW WINDOW
        cv2.imshow(
            "Professional Face Recognition",
            frame
        )

        # EXIT
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # CLEANUP
    video_capture.release()

    cv2.destroyAllWindows()

# ENTRY POINT
if __name__ == "__main__":
    main()

C:\Users\sara\AppData\Roaming\Python\Python313\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename



[INFO] Loading Haaland
[SUCCESS] 01.jpg
[SUCCESS] 02.jpg
[SUCCESS] 03.jpg
[SUCCESS] 04.jpg
[INFO] Haaland loaded with 4 images

[INFO] Loading Hanieh
[SUCCESS] ha01.jpg
[SUCCESS] ha02.jpg
[SUCCESS] ha03.jpg
[SUCCESS] ha04.jpg
[INFO] Hanieh loaded with 4 images

[INFO] Loading Jessie
[SUCCESS] Jessie Pinkman.png
[SUCCESS] jessie2.jpg
[INFO] Jessie loaded with 2 images

[INFO] Loading Raphinha
[SUCCESS] 01.jpg
[SUCCESS] 02.jpg
[SUCCESS] 03.jpg
[SUCCESS] 04.jpg
[INFO] Raphinha loaded with 4 images

[INFO] Loading Rose
[WARNING] No face in rose (2).jpeg
[SUCCESS] rose.jpeg
[SUCCESS] rosee.jpeg
[INFO] Rose loaded with 2 images

[DEBUG] Faces detected: 0

[DEBUG] Faces detected: 1
[DEBUG] Match: Rose (47.5%) | Distance: 0.5249773284885701

[DEBUG] Faces detected: 1
[DEBUG] Match: Rose (50.52%) | Distance: 0.4948181598889379

[DEBUG] Faces detected: 1
[DEBUG] Match: Rose (47.17%) | Distance: 0.5283465369793361

[DEBUG] Faces detected: 1
[DEBUG] Match: Rose (51.57%) | Distance: 0.484340406983